# Bootstrap: how stable is each edge?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/bootstrap.ipynb)

A method fitted to one sample returns one graph, and another sample from the same source would
give a somewhat different one. `andrey.bootstrap` refits the method on many resamples of the rows
and counts how often each edge appears. Here PC runs on data simulated on the ALARM network, whose
true graph we know, to show how to read those counts, what the averaged graph is, and which edges
to trust.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.11.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[viz]==0.11.0" pandas

In [2]:
import warnings

import numpy as np
import pandas as pd

import andrey
import andrey.viz
from andrey.data import load_dataset
from andrey.metrics import shd, to_cpdag

ALARM is a published network of 37 variables and 46 arcs. `load_dataset` simulates 1,000 rows
on it from a linear-Gaussian model and lists the columns in an order drawn from the seed. From
these data PC can recover at best the CPDAG of the true graph, so that is what we score against.

In [3]:
alarm = load_dataset("alarm", n=1000, seed=1)
data, truth = alarm.data, alarm.graph
true_cpdag = to_cpdag(truth)
upper = np.triu_indices(truth.n_nodes, 1)  # each pair of variables once
true_pair = (truth.to_numpy() != 0)[upper]
print(data.shape[0], "rows,", data.shape[1], "variables,", int(true_pair.sum()), "true edges")

1000 rows, 37 variables, 46 true edges


## Edge frequencies rank the edges

`bootstrap` fits PC on 50 resamples, each half of the rows drawn without replacement, the default.
`frequency("absent")` holds the share of the fits in which each pair has no edge, so one minus it
is the share in which the pair is adjacent. Here are the pairs grouped by that share:

In [4]:
out = andrey.bootstrap("pc", data, n_resamples=50, seed=1)
adjacent = (1 - out.frequency("absent"))[upper]

bins = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
rows = []
for low, high in zip(bins[:-1], bins[1:]):
    inside = (adjacent > low) & (adjacent <= high)
    rows.append(
        {
            "adjacent in": f"more than {low:.0%}, up to {high:.0%}",
            "pairs": int(inside.sum()),
            "true pairs": int((inside & true_pair).sum()),
        }
    )
pd.DataFrame(rows)

,adjacent in,pairs,true pairs
0,"more than 10%, up to 30%",6,2
1,"more than 30%, up to 50%",4,3
2,"more than 50%, up to 70%",0,0
3,"more than 70%, up to 90%",2,2
4,"more than 90%, up to 100%",32,31


The pairs adjacent in more than 90% of the fits are nearly all true edges. The shares in between
are not probabilities: most pairs adjacent in 30% to 50% of the fits are true edges here, not 30%
to 50% of them. A frequency says how stable an edge is when the rows change, which ranks the
edges; how often the stable ones are right depends on the method, the data, and the sample
size.

## The averaged graph

`out.structure` is the averaged graph. Each pair takes its most frequent state in the fits, with
no edge counted as one of the states. PC returns CPDAGs, and a vote of CPDAGs is often not a
CPDAG, so the vote is then replaced by the CPDAG of a DAG with the same adjacencies, v-structures,
and directed edges. The result is a valid CPDAG, here with fewer errors than PC on all the rows:

In [5]:
single = andrey.pc(data)
print("single fit:    ", shd(single.structure, true_cpdag), "errors")
print("averaged graph:", shd(out.structure, true_cpdag), "errors")
print("kind:", out.structure.kind, "| built from:", out.metadata["averaged_graph"])
dag = out.structure.to_dag(tie_break="label")
print(
    "the CPDAG of one of its DAGs:",
    np.array_equal(to_cpdag(dag).to_numpy(), out.structure.to_numpy()),
)

single fit:     24 errors
averaged graph: 21 errors
kind: cpdag | built from: vote
the CPDAG of one of its DAGs: True


In [6]:
andrey.viz.draw(out.structure)

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 522 1179" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="522" height="1179" fill="#fafbfc"/><line x1="59.05" y1="612.40" x2="138.65" y2="612.40" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="59.05" y1="270.34" x2="138.65" y2="270.34" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="164.64" y1="352.30" x2="240.83" y2="356.11" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="250.82,356.61 241.08,351.11 240.58,361.10" fill="#3a4658" stroke-linejoin="round"/><line x1="162.35" y1="344.25" x2="253.11" y2="281.47" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="162.26" y1="429.18" x2="245.04" y2="370.55" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="253.20,364.77 242.15,366.47 247.93,374.63" fill="#3a4658" stroke-linejoin="round"/><line x1="164.65" y1="436.80" x2="250.80" y2="437.52" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="162.14" y1="444.38" x2="245.26" y2="505.34" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="253.32,511.25 248.21,501.31 242.30,509.37" fill="#3a4658" stroke-linejoin="round"/><line x1="163.79" y1="617.05" x2="242.33" y2="647.16" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="251.67,650.73 244.12,642.49 240.54,651.82" fill="#3a4658" stroke-linejoin="round"/><line x1="457.70" y1="456.29" x2="386.27" y2="511.03" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="465.63,450.21 454.66,452.33 460.74,460.26" fill="#3a4658" stroke-linejoin="round"/><line x1="453.00" y1="440.80" x2="388.93" y2="436.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="462.98,441.46 453.33,435.81 452.68,445.79" fill="#3a4658" stroke-linejoin="round"/><line x1="59.05" y1="103.05" x2="138.65" y2="103.05" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="161.85" y1="1012.99" x2="245.77" y2="1079.43" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="253.61,1085.64 248.87,1075.51 242.67,1083.35" fill="#3a4658" stroke-linejoin="round"/><line x1="59.05" y1="931.09" x2="138.65" y2="931.09" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="56.70" y1="923.64" x2="132.80" y2="870.44" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="141.00,864.71 129.94,866.34 135.67,874.53" fill="#3a4658" stroke-linejoin="round"/><line x1="245.48" y1="199.19" x2="162.01" y2="262.49" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="253.45,193.15 242.46,195.21 248.50,203.18" fill="#3a4658" stroke-linejoin="round"/><line x1="240.80" y1="185.29" x2="164.65" y2="185.29" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="250.80,185.29 240.80,180.29 240.80,190.29" fill="#3a4658" stroke-linejoin="round"/><line x1="276.20" y1="181.37" x2="354.03" y2="156.72" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="363.56,153.70 352.52,151.96 355.54,161.49" fill="#3a4658" stroke-linejoin="round"/><line x1="276.20" y1="189.22" x2="354.03" y2="213.86" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="363.56,216.88 355.54,209.10 352.52,218.63" fill="#3a4658" stroke-linejoin="round"/><line x1="164.65" y1="270.77" x2="250.81" y2="273.65" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="58.37" y1="1115.61" x2="139.33" y2="1142.83" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="58.37" y1="1107.32" x2="139.33" y2="1080.09" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="353.80" y1="1090.54" x2="164.18" y2="1143.48" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="363.43,1087.86 352.46,1085.73 355.14,1095.36" fill="#3a4658" stroke-linejoin="round"/><line x1="355.80"

## Subsampling or the bootstrap

`resampling="bootstrap"` draws as many rows as the data have, with replacement, instead of half of
them without. A repeated row makes a test see more evidence than the resample holds, so weak and
false edges survive in more of the fits. `sample_fraction` sets the share of the rows in a
subsample: a larger share gives fits closer to the fit on all the rows, so the fits agree more and
their frequencies say less about how much the result varies.

In [7]:
runs = {
    "subsample, 50% of the rows": {},
    "subsample, 80% of the rows": {"sample_fraction": 0.8},
    "bootstrap": {"resampling": "bootstrap"},
}
results, raised = {}, {}
rows = []
for name, settings in runs.items():
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = andrey.bootstrap("pc", data, n_resamples=50, seed=1, **settings)
    results[name] = result
    raised[name] = [str(w.message) for w in caught if issubclass(w.category, andrey.AndreyWarning)]
    share = (1 - result.frequency("absent"))[upper]
    above, middle = share > 0.9, (share > 0.5) & (share <= 0.9)
    rows.append(
        {
            "resampling": name,
            "errors": shd(result.structure, true_cpdag),
            "above 90%: true of all": f"{(above & true_pair).sum()} of {above.sum()}",
            "50% to 90%: true of all": f"{(middle & true_pair).sum()} of {middle.sum()}",
            "averaged graph": result.metadata["averaged_graph"],
        }
    )
pd.DataFrame(rows)

,resampling,errors,above 90%: true of all,50% to 90%: true of all,averaged graph
0,"subsample, 50% of the rows",21,31 of 32,2 of 2,vote
1,"subsample, 80% of the rows",15,33 of 34,4 of 5,vote
2,bootstrap,19,32 of 33,4 of 5,nearest_resample


## When the vote has no valid graph

With the bootstrap, the vote of these fits has no consistent extension: no DAG keeps all its
adjacencies, v-structures, and directed edges, so no CPDAG holds the vote. `bootstrap` then
returns the resample graph nearest the others among those a DAG extends, the fit whose state of
each pair the most fits share. It is a graph PC itself returned, and a warning and
`metadata["averaged_graph"]` say so:

In [8]:
boot = results["bootstrap"]
print("\n".join(raised["bootstrap"]))
print("built from:", boot.metadata["averaged_graph"], "| kind:", boot.structure.kind)

the vote of the fits has no valid cpdag: PDAG admits no consistent DAG extension: no DAG has its skeleton, its v-structures, and its directed edges, because the undirected edges among nodes [2, 3, 4, 5, 11, 13, 15, 29, 30, 31, 32] cannot be oriented without adding a v-structure or closing a directed cycle. The averaged graph is resample 25, the valid fit nearest the others; strict=True raises instead
built from: nearest_resample | kind: cpdag


`strict=True` raises instead, so you can decide from the frequencies, which keep the whole vote
either way:

In [9]:
try:
    andrey.bootstrap("pc", data, n_resamples=50, seed=1, resampling="bootstrap", strict=True)
except ValueError as error:
    print("ValueError:", error)

ValueError: the vote of the fits has no valid cpdag: PDAG admits no consistent DAG extension: no DAG has its skeleton, its v-structures, and its directed edges, because the undirected edges among nodes [2, 3, 4, 5, 11, 13, 15, 29, 30, 31, 32] cannot be oriented without adding a v-structure or closing a directed cycle. strict=False returns the fit nearest the others instead


## Which edges to trust, and how to report them

Trust the edges most fits agree on, and report each one with its frequency. Here are the averaged
graph's most and least stable edges, each with the share of the fits in which the pair is adjacent
and the share with this edge:

In [10]:
position = {name: k for k, name in enumerate(data.columns)}
glyph = {"directed": "->", "undirected": "--"}
pairs = 1 - out.frequency("absent")
report = pd.DataFrame(
    {
        "edge": f"{source} {glyph[kind]} {target}",
        "adjacent": pairs[position[source], position[target]],
        "this edge": out.frequency(glyph[kind])[position[source], position[target]],
        "true pair": bool(truth.to_numpy()[position[source], position[target]]),
    }
    for source, target, kind in out.structure.oriented_edges()
).sort_values("adjacent", ascending=False, kind="stable")
pd.concat([report.head(4), report.tail(4)])

,edge,adjacent,this edge,true pair
0,PAP -- PULMEMBOLUS,1.00,0.90,True
1,HISTORY -- LVFAILURE,1.00,1.00,True
2,CO -> BP,1.00,0.60,True
4,TPR -> BP,1.00,0.60,True
26,VENTTUBE -> VENTLUNG,0.98,0.82,True
32,INTUBATION -> SHUNT,0.98,0.92,True
3,CO -- STROKEVOLUME,0.80,0.68,True
14,LVFAILURE -> LVEDVOLUME,0.80,0.78,True


`threshold` asks more of the averaged graph: `out.average(0.9)` keeps only the pairs adjacent in
more than 90% of the fits, without refitting, which here drops true edges as well as false ones.
With each edge, report what produced the frequencies:
the method and its settings, the number of resamples, the resampling and its share of rows, the
seed, the threshold, and whether the averaged graph is the vote or the nearest resample.

In [11]:
strong = out.average(0.9)
print(len(strong.oriented_edges()), "edges,", shd(strong, true_cpdag), "errors")
print(
    {
        "method": out.metadata["method"],
        "n_resamples": out.n_resamples,
        "threshold": out.threshold,
        **{
            key: out.metadata[key]
            for key in ("resampling", "sample_fraction", "seed", "averaged_graph")
        },
    }
)

32 edges, 25 errors
{'method': 'pc', 'n_resamples': 50, 'threshold': 0.5, 'resampling': 'subsample', 'sample_fraction': 0.5, 'seed': 1, 'averaged_graph': 'vote'}
